# Jyotish LoRA v4: rule paraphrases on a Kaggle T4

Runs one shard of `src/dataset/rule_qa.py generate` against llama.cpp serving Qwen3-14B Q4_K_M. Output: `samples_<shard>.jsonl` in the kernel output; download it and run `python -m src.dataset.rule_qa merge-samples <file>` on the workstation.

In [ ]:
import glob, json, os, subprocess, sys, tarfile, time, urllib.request, hashlib, zipfile
os.environ["CUDA_VISIBLE_DEVICES"] = "0"       # one GPU per kernel

# ---------- settings ----------
SHARDS = "4/8"                 # which shards of the rule base this kernel takes, e.g. "4/8" or "4,5/8"
K = 4                          # pairs per rule
PARALLEL = 4                   # llama-server slots = generator workers
MAX_HOURS = 11.3               # stop and save before Kaggle's 12 h limit; rerun with the same SHARDS to continue
RELEASE = "b11513"
MODEL_REPO = "unsloth/Qwen3-14B-GGUF"
MODEL_FILE = "Qwen3-14B-Q4_K_M.gguf"
CTX_PER_SLOT = 4096
PORT = 8080
WORK = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
PREBUILT = {
    "llama-b11513-bin-ubuntu-cuda-12.8-x64.tar.gz": "af399b958079ec0a0a002faa5b2432f63269c0bf862e9f4f8add67a38a17aa33",
    "cudart-llama-b11513-bin-ubuntu-cuda-12.8-x64.tar.gz": "bb8ac07d9ae02a9d834d03cd5b2ed7c3b9720f46aea02414d0c304d066fec13c",
}

In [ ]:
def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print((r.stdout + r.stderr)[-2000:], flush=True)
    if check:
        assert r.returncode == 0, "command failed: " + cmd
    return r.returncode, r.stdout + r.stderr

print(sh("nvidia-smi")[1][:700])
sh(f"{sys.executable} -m pip install -q datasketch langdetect huggingface_hub pyswisseph")   # pyswisseph: belt and braces for the chart imports

# repo code -> /kaggle/working/repo. Kaggle sometimes unpacks code.zip on upload (then src/ sits under a code/ folder).
import shutil
REPO = "/kaggle/working/repo"
zips = glob.glob("/kaggle/input/**/code.zip", recursive=True)
if zips:
    zipfile.ZipFile(zips[0]).extractall(REPO)
else:
    marker = glob.glob("/kaggle/input/**/src/dataset/rule_qa.py", recursive=True)
    assert marker, "neither code.zip nor an unpacked code/src folder found in the inputs"
    base = os.path.dirname(os.path.dirname(os.path.dirname(marker[0])))
    for sub in ("src", "prompts"):
        shutil.copytree(os.path.join(base, sub), os.path.join(REPO, sub), dirs_exist_ok=True)
print("repo code at", REPO, os.listdir(REPO))
RULES = glob.glob("/kaggle/input/**/rules.jsonl", recursive=True)[0]
DONE = glob.glob("/kaggle/input/**/done_rule_ids.txt", recursive=True)
print("rules:", RULES, "| done list:", DONE)

# llama.cpp prebuilt CUDA build (verified sha256), fall back to a source build
def fetch(name, sha):
    d = WORK + "/llama_prebuilt"; os.makedirs(d, exist_ok=True)
    path = d + "/" + name
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"https://github.com/ggml-org/llama.cpp/releases/download/{RELEASE}/{name}", path)
    assert hashlib.sha256(open(path, "rb").read()).hexdigest() == sha, "checksum mismatch: " + name
    tarfile.open(path).extractall(d)
    return d
for name, sha in PREBUILT.items():
    d = fetch(name, sha)
SERVER = glob.glob(d + "/**/llama-server", recursive=True)[0]
LIBS = os.path.dirname(SERVER)
os.environ["LD_LIBRARY_PATH"] = LIBS + ":" + os.environ.get("LD_LIBRARY_PATH", "")
rc, out = sh(f"chmod +x {SERVER} && {SERVER} --version", check=False)
if rc != 0:
    print("prebuilt build does not run here: building from source (about 10 min)")
    src = WORK + "/llama.cpp"
    sh(f"git clone --depth 1 --branch {RELEASE} https://github.com/ggml-org/llama.cpp {src}")
    sh(f"cd {src} && cmake -B build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release && cmake --build build --target llama-server -j4")
    SERVER = src + "/build/bin/llama-server"
print("llama.cpp ready:", SERVER)

from huggingface_hub import hf_hub_download
MODEL = hf_hub_download(MODEL_REPO, MODEL_FILE, local_dir=WORK + "/model")
print("model:", MODEL, round(os.path.getsize(MODEL) / 2**30, 1), "GiB")

In [ ]:
server_log = open("/kaggle/working/llama_server.log", "w")
server = subprocess.Popen([SERVER, "-m", MODEL, "-ngl", "99", "-c", str(CTX_PER_SLOT * PARALLEL), "-np", str(PARALLEL),
                           "--port", str(PORT), "--host", "127.0.0.1", "-fa", "on", "--jinja"], stdout=server_log, stderr=subprocess.STDOUT)
t0 = time.time()
while True:
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5).read()
        break
    except Exception:
        time.sleep(5)
    assert server.poll() is None, "llama-server exited: see /kaggle/working/llama_server.log"
    assert time.time() - t0 < 900, "llama-server did not become ready"
print("server ready in %.0f s" % (time.time() - t0))

In [ ]:
# the repo's own generator and checks, against the local llama-server
shard_tag = SHARDS.replace("/", "of").replace(",", "-")
OUT = f"/kaggle/working/samples_{shard_tag}.jsonl"
cmd = [sys.executable, "-X", "utf8", "-m", "src.dataset.rule_qa", "generate", "--backend", "llamacpp",
       "--host", f"http://127.0.0.1:{PORT}", "--model", MODEL_FILE, "--k", str(K), "--workers", str(PARALLEL),
       "--shards", SHARDS, "--rules", RULES, "--samples", OUT, "--max-hours", str(MAX_HOURS)]
if DONE:
    # rules the workstation already sampled are skipped: the done list is a tiny samples-like file (rule_id per line)
    done_file = "/kaggle/working/done_from_workstation.jsonl"
    with open(done_file, "w") as f:
        for rid in open(DONE[0]).read().split():
            f.write(json.dumps({"rule_id": rid, "pairs": []}) + "\n")
    cmd += ["--done-from", done_file]
print(" ".join(cmd), flush=True)
r = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True)
print(r.stdout[-6000:]); print(r.stderr[-3000:])
server.terminate()
assert r.returncode == 0, "generator failed (see stderr above)"   # so a broken run shows as an error, not 'complete'
n = sum(1 for _ in open(OUT)) if os.path.exists(OUT) else 0
json.dump({"shards": SHARDS, "k": K, "model": MODEL_FILE, "release": RELEASE, "rules_sampled": n},
          open("/kaggle/working/rule_qa_manifest.json", "w"), indent=2)
print("DONE:", n, "rules sampled ->", OUT)